In [ ]:
import sys
from pathlib import Path

current_dir = Path.cwd().resolve()

# Find 'src' by walking up the directory tree
src_path = None
for parent in [current_dir] + list(current_dir.parents):
    candidate = parent / 'src'
    if candidate.exists() and candidate.is_dir():
        src_path = candidate
        break

if src_path:
    if str(src_path) not in sys.path:
        sys.path.append(str(src_path))
    print(f"Success: Linked to src at {src_path}")
    
    # Import
    try:
        from config import *
        print(f"Environment linked. Data dir: {DATA_DIR}")    
    except ImportError as e:
        print(f"CRITICAL: Found src but could not import config. {e}")
else:
    print("CRITICAL ERROR: Could not find 'src' directory in any parent folder.")
    print(f"Current search path: {current_dir}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from plotting import *
setup_plotting_style()
print(f"saving figures:{SAVE_FIGURES}")

In [ ]:
NB_ID = "13"
SAMPLES_TO_PLOT = 5000
ROWS_TO_CHECK = [0, 50]

In [ ]:
def plot_row_verification(raw_row, clean_row, signal_name, row_idx):
    """
    Plots a 2-panel comparison. 
    """
    # Calculate Stats
    raw_mean = np.mean(raw_row)
    clean_mean = np.mean(clean_row)
    shift_distance = np.abs(raw_mean - clean_mean)
    
    # Subset for Plotting
    raw_sub = raw_row[:SAMPLES_TO_PLOT]
    
    # Setup Figure
    fig = plt.figure(figsize=(14, 7))
    fig.suptitle(f"DC Verification: {signal_name} | Row {row_idx}", fontsize=16)
    
    # --- Panel 1: Full Constellation ---
    ax0 = fig.add_subplot(1, 2, 1)
    ax0.scatter(raw_sub.real, raw_sub.imag, s=1, alpha=0.3, color='gray', label='Signal Cloud')
    ax0.scatter(raw_mean.real, raw_mean.imag, s=150, marker='X', color='C0', edgecolors='black', label='Raw Mean', zorder=10)
    ax0.scatter(clean_mean.real, clean_mean.imag, s=150, marker='P', color='C1', edgecolors='black', label='Clean Mean', zorder=10)
    ax0.set_title("1. Full Constellation")
    ax0.set_xlabel("In-Phase")
    ax0.set_ylabel("Quadrature")
    ax0.axis('equal')
    ax0.grid(True, alpha=0.3)

    # --- Panel 2: The Microscope ---
    ax1 = fig.add_subplot(1, 2, 2)
    zoom_radius = max(shift_distance * 4, 0.01) 
    
    # Plot Means (Keep size large here for visibility)
    ax1.scatter(raw_mean.real, raw_mean.imag, s=400, marker='X', color='C0', label='Raw Center', zorder=5)
    ax1.scatter(clean_mean.real, clean_mean.imag, s=400, marker='P', color='C1', label='Clean Center', zorder=5)
    
    ax1.annotate("", 
                 xy=(clean_mean.real, clean_mean.imag), 
                 xytext=(raw_mean.real, raw_mean.imag),
                 arrowprops=dict(arrowstyle="->", color="black", lw=2, linestyle="--"))
    
    ax1.set_xlim(-zoom_radius, zoom_radius)
    ax1.set_ylim(-zoom_radius, zoom_radius)
    ax1.set_title(f"2. Center Zoom (Radius: {zoom_radius:.4f})")
    ax1.grid(True, alpha=0.5)
    ax1.axhline(0, color='black', alpha=0.3)
    ax1.axvline(0, color='black', alpha=0.3)
    
    # --- FIXED LEGEND ---
    handles, labels = ax1.get_legend_handles_labels()
    fig.legend(handles, labels, 
               loc='lower center', 
               ncol=2, 
               bbox_to_anchor=(0.5, 0.02), 
               fontsize=12,
               
               # THE FIXES:
               markerscale=0.6,       # Shrink markers to 50% size (matches text better)
               handletextpad=0.5,     # More space between marker and text
               columnspacing=2.0,     # More space between the two columns
               borderpad=0.8,         # More padding inside the box border
               frameon=True           # Ensure the box is visible
              )
    
    plt.tight_layout(rect=[0, 0.1, 1, 0.95])
    
    # Save
    slug = f"verify_dc_zoom_{signal_name}_row{row_idx:03d}"
    save_plot(slug, machine_id="B", nb_id=NB_ID, fig_id=f"01")
    plt.show()

In [ ]:
for signal in SIGNALS:
    # Generate paths using the new system
    raw_path = get_unet_path(STAGE_RAW, signal=signal)
    clean_path = get_unet_path(STAGE_CLEANED, signal=signal)
    
    # Safety check: Ensure both files exist
    if not raw_path.exists() or not clean_path.exists():
        print(f"Skipping {signal}: Missing raw or cleaned files.")
        continue
        
    print(f"--- Verifying {signal} ---")
    
    # Load matrices
    raw_matrix = np.load(raw_path)
    clean_matrix = np.load(clean_path)
    
    # Iterate through requested rows
    for row_idx in ROWS_TO_CHECK:
        if row_idx < clean_matrix.shape[0]:
            print(f"Plotting Row {row_idx}...")
            
            # Extract 1D arrays
            raw_row = raw_matrix[row_idx]
            clean_row = clean_matrix[row_idx]
            
            # Plot
            plot_row_verification(
                raw_row, 
                clean_row, 
                signal, 
                row_idx, 
            )
        else:
            print(f"Row {row_idx} out of bounds for {signal} (Max: {clean_matrix.shape[0]-1})")